# Ejercicio 3 — Transformer Decoder para predicción de tokens

En este ejercicio vamos a construir un **decoder Transformer básico** para predecir el siguiente token de una secuencia.

La idea general es entrenar el modelo de forma **autoregresiva**:

- A partir de una secuencia de entrada, el modelo aprende a predecir el siguiente token.
- Se usa **causal masking** para que el modelo no pueda mirar tokens futuros.
- La salida no es una única clase final, sino una predicción de vocabulario para cada posición de la secuencia.

Este ejercicio se diferencia del clasificador Transformer anterior porque aquí no queremos clasificar una frase completa, sino **generar texto token a token**.

## 1. Importación de librerías

Usaremos TensorFlow/Keras para construir el modelo, `Tokenizer` para convertir palabras en números y `pad_sequences` para igualar la longitud de todas las secuencias.

In [3]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.layers import (
    Input,
    Embedding,
    MultiHeadAttention,
    LayerNormalization,
    Dense,
    Dropout
)
from tensorflow.keras.models import Model

# Fijamos semilla para que los resultados sean algo más reproducibles
np.random.seed(42)
tf.random.set_seed(42)

## 2. Conjunto de frases

El enunciado proporciona un conjunto pequeño de frases en español. El objetivo es que el modelo aprenda patrones sencillos como:

- `El caballo` → puede continuar con `trota`, `galopa`, `corre`, etc.
- `El perro` → puede continuar con `corre`, `ladra`, `juega`, etc.

Como el dataset es muy pequeño, no esperamos una generación perfecta, sino entender el funcionamiento del decoder.

In [4]:
sentences = [
    "El perro corre feliz",
    "El gato salta ágil",
    "La tortuga camina lenta",
    "El caballo trota fuerte",
    "El perro ladra ruidoso",
    "El gato duerme tranquilo",
    "La tortuga nada lenta",
    "El caballo galopa veloz",
    "El perro juega contento",
    "El gato observa curioso",
    "La tortuga descansa pacífica",
    "El caballo relincha bravo",
    "El perro huele atento",
    "El gato maúlla suave",
    "La tortuga explora cautelosa",
    "El caballo corre elegante"
]

for sentence in sentences[:5]:
    print(sentence)

El perro corre feliz
El gato salta ágil
La tortuga camina lenta
El caballo trota fuerte
El perro ladra ruidoso


## 3. Parámetros del ejercicio

Según el enunciado:

- Tamaño del vocabulario: `40` tokens.
- Longitud máxima de secuencia: `10` tokens.
- Dimensión de embeddings: `32`.

El `vocab_size` indica cuántos tokens diferentes puede representar el modelo.

El `max_length` indica que todas las secuencias tendrán exactamente 10 posiciones tras aplicar padding.

El `embedding_dim` indica el tamaño del vector que representa cada token.

In [5]:
vocab_size = 40
max_length = 10
embedding_dim = 32

print("vocab_size:", vocab_size)
print("max_length:", max_length)
print("embedding_dim:", embedding_dim)

vocab_size: 40
max_length: 10
embedding_dim: 32


## 4. Tokenización

La tokenización convierte cada palabra en un número entero.

Por ejemplo:

```text
"El perro corre feliz"
```

puede convertirse en algo parecido a:

```text
[1, 2, 3, 4]
```

El modelo no trabaja directamente con palabras, sino con estos identificadores numéricos.

In [6]:
tokenizer = Tokenizer(
    num_words=vocab_size,
    filters='',          # no eliminamos caracteres automáticamente
    lower=False,         # mantenemos mayúsculas/minúsculas como en las frases
    oov_token='<OOV>'    # token para palabras fuera del vocabulario
)

tokenizer.fit_on_texts(sentences)

sequences = tokenizer.texts_to_sequences(sentences)

print("Vocabulario aprendido:")
print(tokenizer.word_index)

print("\nEjemplo de frase tokenizada:")
print(sentences[0], "->", sequences[0])

Vocabulario aprendido:
{'<OOV>': 1, 'El': 2, 'perro': 3, 'gato': 4, 'La': 5, 'tortuga': 6, 'caballo': 7, 'corre': 8, 'lenta': 9, 'feliz': 10, 'salta': 11, 'ágil': 12, 'camina': 13, 'trota': 14, 'fuerte': 15, 'ladra': 16, 'ruidoso': 17, 'duerme': 18, 'tranquilo': 19, 'nada': 20, 'galopa': 21, 'veloz': 22, 'juega': 23, 'contento': 24, 'observa': 25, 'curioso': 26, 'descansa': 27, 'pacífica': 28, 'relincha': 29, 'bravo': 30, 'huele': 31, 'atento': 32, 'maúlla': 33, 'suave': 34, 'explora': 35, 'cautelosa': 36, 'elegante': 37}

Ejemplo de frase tokenizada:
El perro corre feliz -> [2, 3, 8, 10]


## 5. Creación de secuencias de entrada `X` y salida `Y`

Para entrenar el decoder de forma autoregresiva, cada frase se divide así:

Frase original:

```text
[w1, w2, w3, w4]
```

Entrada del modelo:

```text
X = [w1, w2, w3]
```

Salida esperada:

```text
Y = [w2, w3, w4]
```

Es decir, el modelo aprende que en cada posición debe predecir el token siguiente.

In [7]:

X = []
Y = []

for seq in sequences:
    X.append(seq[:-1])  # frase sin el último token
    Y.append(seq[1:])   # frase sin el primer token

print("Ejemplo original:", sequences[0])
print("X:", X[0])
print("Y:", Y[0])

Ejemplo original: [2, 3, 8, 10]
X: [2, 3, 8]
Y: [3, 8, 10]


## 6. Padding

Las frases pueden tener longitudes distintas, pero el modelo necesita que todas las entradas tengan la misma longitud.

Por eso usamos `pad_sequences` con `maxlen=max_length`.

En este ejercicio usamos el padding por defecto, que añade ceros al principio de la secuencia.

El token `0` representa padding.

In [8]:
X = pad_sequences(X, maxlen=max_length)
Y = pad_sequences(Y, maxlen=max_length)

X = tf.convert_to_tensor(X, dtype=tf.int32)
Y = tf.convert_to_tensor(Y, dtype=tf.int32)

print("Forma de X:", X.shape)
print("Forma de Y:", Y.shape)

print("\nEjemplo X con padding:")
print(X[0].numpy())

print("\nEjemplo Y con padding:")
print(Y[0].numpy())

Forma de X: (16, 10)
Forma de Y: (16, 10)

Ejemplo X con padding:
[0 0 0 0 0 0 0 2 3 8]

Ejemplo Y con padding:
[ 0  0  0  0  0  0  0  3  8 10]


## 7. Modelo decoder Transformer

Ahora construimos el modelo.

La diferencia clave respecto al Transformer clasificador es esta línea:

```python
use_causal_mask=True
```

Esto impide que el modelo vea tokens futuros durante el entrenamiento.

Además, no usamos `GlobalAveragePooling1D`, porque ahora queremos una predicción para cada posición de la secuencia.

La salida final es:

```python
Dense(vocab_size)
```

Esto significa que, para cada posición, el modelo devuelve una puntuación para cada token posible del vocabulario.

In [9]:
def create_transformer_decoder(vocab_size, max_length, embedding_dim):
    inputs = Input(shape=(max_length,), dtype=tf.int32)

    # Capa de embedding: cada token pasa a ser un vector de embedding_dim dimensiones
    embedding_layer = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim
    )(inputs)

    # Self-attention causal: el modelo solo puede atender a tokens anteriores o actuales
    attention = MultiHeadAttention(
        num_heads=2,
        key_dim=embedding_dim
    )(
        embedding_layer,
        embedding_layer,
        embedding_layer,
        use_causal_mask=True
    )

    # Conexión residual + normalización
    x = LayerNormalization()(attention + embedding_layer)

    # Pequeño dropout para regularizar
    x = Dropout(0.1)(x)

    # Predicción del vocabulario en cada posición
    outputs = Dense(vocab_size)(x)

    return Model(inputs, outputs)


model = create_transformer_decoder(
    vocab_size=vocab_size,
    max_length=max_length,
    embedding_dim=embedding_dim
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 10, 32)    │      1,280 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 32)    │      8,416 │ embedding[0][0],  │
│ (MultiHeadAttentio… │                   │            │ embedding[0][0],  │
│                     │                   │            │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 32)    │          0 │ multi_head_atten… │
│                     │                   │            │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 10, 32)    │         64 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 10, 32)    │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 10, 40)    │      1,320 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 11,080 (43.28 KB)

 Trainable params: 11,080 (43.28 KB)

 Non-trainable params: 0 (0.00 B)

## 8. Compilación del modelo

Como ahora la salida es una distribución sobre el vocabulario, usamos:

```python
SparseCategoricalCrossentropy(from_logits=True)
```

Usamos `from_logits=True` porque la última capa `Dense(vocab_size)` no tiene función `softmax`.

In [10]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

## 9. Entrenamiento

Entrenamos durante bastantes épocas porque el dataset es muy pequeño.

En un problema real no sería recomendable memorizar, pero aquí el objetivo es que el modelo aprenda el mecanismo de predicción autoregresiva.

In [11]:
history = model.fit(
    X,
    Y,
    epochs=300,
    batch_size=4,
    verbose=1
)

Epoch 1/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3375 - loss: 3.1912   
Epoch 2/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7063 - loss: 1.7832 
Epoch 3/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7000 - loss: 1.4221 
Epoch 4/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7000 - loss: 1.2730 
Epoch 5/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7000 - loss: 1.1925 
Epoch 6/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7188 - loss: 1.1258 
Epoch 7/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7625 - loss: 1.0286 
Epoch 8/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7688 - loss: 0.9595 
Epoch 9/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8000 - loss: 0.8675 
Epoch 10/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7938 - loss: 0.8410 
Epoch 11/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8250 - loss: 0.7944 
Epoch 12/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8188 - l

## 10. Función para predecir el siguiente token

Dada una secuencia parcial, la función:

1. Tokeniza la frase.
2. Aplica padding.
3. Obtiene las predicciones del modelo.
4. Toma la predicción de la última posición.
5. Devuelve el token con mayor probabilidad.

La última posición es importante porque representa el siguiente token que queremos generar tras la secuencia dada.

In [12]:
def predict_next_token(model, tokenizer, sentence, max_length):
    seq = tokenizer.texts_to_sequences([sentence])[0]
    seq = pad_sequences([seq], maxlen=max_length)

    predictions = model.predict(seq, verbose=0)

    # Tomamos la predicción correspondiente a la última posición
    last_pred_logits = predictions[0, -1]

    # Seleccionamos el token con mayor puntuación
    next_token_id = np.argmax(last_pred_logits)

    return tokenizer.index_word.get(next_token_id, None)


print("Siguiente token para 'El caballo':")
print(predict_next_token(model, tokenizer, "El caballo", max_length))

Siguiente token para 'El caballo':
corre


## 11. Generación de varios tokens

Ahora generamos tokens de forma iterativa:

1. Partimos de una frase inicial.
2. Predecimos el siguiente token.
3. Añadimos ese token a la frase.
4. Repetimos el proceso.

El enunciado indica generar hasta llegar a `None`. Añadimos también un límite máximo de tokens para evitar bucles infinitos.

In [13]:
def generate_text(model, tokenizer, start_sentence, max_length, max_new_tokens=10):
    generated = start_sentence

    for _ in range(max_new_tokens):
        next_token = predict_next_token(model, tokenizer, generated, max_length)

        if next_token is None:
            break

        generated += " " + next_token

    return generated


print(generate_text(model, tokenizer, "El caballo", max_length))

El caballo corre elegante corre elegante corre elegante corre elegante tortuga explora


## 12. Pruebas con distintas secuencias

Probamos con varias frases iniciales para observar cómo se comporta el modelo.

Como el dataset es muy pequeño, es normal que el modelo tienda a memorizar patrones del entrenamiento.

In [14]:
test_sentences = [
    "El caballo",
    "El perro",
    "El gato",
    "La tortuga",
    "El caballo corre",
    "El gato observa"
]

for sentence in test_sentences:
    generated = generate_text(model, tokenizer, sentence, max_length, max_new_tokens=5)
    print(f"Entrada: {sentence}")
    print(f"Generado: {generated}")
    print("-" * 50)

Entrada: El caballo
Generado: El caballo corre elegante corre elegante corre
--------------------------------------------------
Entrada: El perro
Generado: El perro corre feliz
--------------------------------------------------
Entrada: El gato
Generado: El gato duerme tranquilo corre elegante tortuga
--------------------------------------------------
Entrada: La tortuga
Generado: La tortuga nada lenta contento elegante tortuga
--------------------------------------------------
Entrada: El caballo corre
Generado: El caballo corre elegante corre elegante corre elegante
--------------------------------------------------
Entrada: El gato observa
Generado: El gato observa curioso
--------------------------------------------------


## 13. Conclusión

En este ejercicio hemos construido un **decoder Transformer básico** para predicción autoregresiva de tokens.

La parte más importante es el uso de `use_causal_mask=True`, que evita que el modelo use información del futuro durante el entrenamiento.

A diferencia del Transformer clasificador, aquí no usamos `GlobalAveragePooling1D`, porque no queremos resumir toda la frase en un único vector. En su lugar, mantenemos una salida por cada posición de la secuencia y usamos una capa `Dense(vocab_size)` para predecir el token siguiente.

Como el conjunto de entrenamiento es muy pequeño, los resultados pueden variar y el modelo puede memorizar frases concretas. Aun así, el ejercicio permite entender la lógica básica de un decoder autoregresivo.